In [ ]:
import matplotlib.pyplot as plt
import re
import gzip
import shutil
import pandas as pd
import scipy.sparse
from multiprocessing import Pool
import tarfile
import seaborn as sns
import scanpy.external as sce 
import harmonypy as hm
import anndata as ad
import omicverse as ov
import scvi
import scib
import annoy
import gc
import numpy as np
import random
import scanpy as sc
import os

In [ ]:
adata_3mono = adata_mono.copy()

In [ ]:
import re
import numpy as np

def extract_day(x):
    x = str(x)
    m = re.match(r"^D(-?\d+(?:\.\d+)?)_batch\d+$", x)
    if m is None:
        return np.nan
    return float(m.group(1))

adata_3mono.obs["day"] = (
    adata_3mono.obs["batch"]
    .astype(str)
    .map(extract_day)
)

In [ ]:
adata_3mono.obs[
    ["batch", "day"]
].drop_duplicates().sort_values("day")

In [ ]:
adata_3mono = adata_3mono[
    adata_3mono.obs["source"] == "xenotransplantation"
].copy()

In [ ]:
print(adata_3mono.obs["batch"].value_counts())
print(adata_3mono.obs["day"].isna().sum())

In [ ]:
import pymfuzz

In [ ]:
adata_3mono.X = adata_3mono.layers["counts"].copy()

print(
    "X min/max:",
    adata_3mono.X.min(),
    adata_3mono.X.max()
)

In [ ]:
adata_pf4 = adata_3mono[
    adata_3mono.obs["mono_subtype"] == "PF4+PPBP+ Mono"
].copy()

print(adata_pf4)

print(
    adata_pf4.obs.groupby("day", observed=True)
    .size()
    .sort_index()
)

In [ ]:
import scipy.sparse as sp

days = np.sort(
    adata_pf4.obs["day"].astype(float).unique()
)

pb_counts = []
pb_ncells = []

for day in days:

    mask = (
        adata_pf4.obs["day"].astype(float).values
        == day
    )

    X = adata_pf4[mask].X

    if sp.issparse(X):
        x_sum = np.asarray(
            X.sum(axis=0)
        ).ravel()
    else:
        x_sum = np.asarray(X).sum(axis=0)

    pb_counts.append(x_sum)
    pb_ncells.append(mask.sum())

In [ ]:
days = np.sort(
    adata_pf4.obs["day"].astype(float).unique()
)

pb_counts = []
pb_ncells = []

for day in days:

    mask = (
        adata_pf4.obs["day"].astype(float).values
        == day
    )

    X = adata_pf4[mask].X

    if sp.issparse(X):
        x_sum = np.asarray(
            X.sum(axis=0)
        ).ravel()
    else:
        x_sum = np.asarray(X).sum(axis=0)

    pb_counts.append(x_sum)
    pb_ncells.append(mask.sum())

In [ ]:
time_names = [
    f"D{x:g}" for x in days
]

adata_pf4_pb = ad.AnnData(
    X=np.vstack(pb_counts),
    obs=pd.DataFrame(
        {
            "day": days,
            "n_cells": pb_ncells
        },
        index=time_names
    ),
    var=adata_pf4.var.copy()
)

In [ ]:
counts = ov.pd.DataFrame(
    adata_pf4_pb.X.T,
    index=adata_pf4_pb.var_names,
    columns=adata_pf4_pb.obs_names
)

In [ ]:
keep = (counts >= 10).sum(axis=1) >= 3
counts = counts.loc[keep]

print(
    counts.shape[0],
    "genes pass the low-count filter"
)

In [ ]:
norm = ov.bulk.deseq2_normalize(counts)
lognorm = ov.np.log1p(norm)

display(lognorm.iloc[:3, :4])

In [ ]:
time = adata_pf4_pb.obs["day"].astype(float)

In [ ]:
assert list(counts.columns) == list(time.index)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

hv = lognorm.loc[
    lognorm.var(axis=1).sort_values().index[-2000:]
]

# PCA
pcs = PCA(
    n_components=5,
    random_state=0
).fit(hv.T.values)

emb = pcs.transform(
    hv.T.values
)

pc_var = pcs.explained_variance_ratio_ * 100

axlab = dict(
    xlabel=f"PC1 ({pc_var[0]:.0f}%)",
    ylabel=f"PC2 ({pc_var[1]:.0f}%)"
)

print(
    f"PCA — PC1 {pc_var[0]:.0f}%, "
    f"PC2 {pc_var[1]:.0f}%, "
    f"PC3 {pc_var[2]:.0f}% of variance"
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(5.5, 4.5)
)

sc = ax.scatter(
    emb[:, 0],
    emb[:, 1],
    c=time.values,
    cmap="viridis",
    s=70,
    edgecolor="k",
    linewidth=0.3
)

plt.colorbar(
    sc,
    ax=ax,
    label="Day"
)

ax.set(
    title="PF4+PPBP+ Mono pseudobulk",
    **axlab
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 5)
)

sc = ax.scatter(
    emb[:, 0],
    emb[:, 1],
    c=time.values,
    cmap="viridis",
    s=70,
    edgecolor="k",
    linewidth=0.3
)

for i, sample in enumerate(lognorm.columns):
    ax.annotate(
        sample,
        (emb[i, 0], emb[i, 1]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )

plt.colorbar(
    sc,
    ax=ax,
    label="Day"
)

ax.set(
    title="PF4+PPBP+ Mono pseudobulk",
    **axlab
)

plt.tight_layout()
plt.show()

In [ ]:
deg = ov.bulk.pyDEG(counts)
deg.drop_duplicates_index()

res = deg.timecourse_deg(
    time=time,
    data_type="counts",
    time_basis="spline",
    spline_df=3
)

In [ ]:
res[["F", "pvalue", "qvalue", "sig"]].head()

In [ ]:
temporal = list(
    res.index[
        res["sig"] == "temporal"
    ]
)

print(
    len(temporal),
    "of",
    res.shape[0],
    "genes are temporally regulated (q < 0.05)"
)

In [ ]:
import scipy.stats as st
obs = res["F"].sort_values().values

quant = (
    np.arange(1, len(obs) + 1) - 0.5
) / len(obs)

theo = st.f.ppf(
    quant,
    3,
    11
)

print(
    f"Q-Q input: {len(obs)} genes; "
    f"observed moderated-F spans "
    f"{obs[0]:.2f}..{obs[-1]:.1f}"
)

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(
    theo,
    obs,
    s=6,
    alpha=0.5
)

lim = np.nanpercentile(theo, 99.5)

ax.plot(
    [0, lim],
    [0, lim],
    "k--",
    lw=1
)

ax.set(
    xlim=(0, lim),
    ylim=(0, np.nanpercentile(obs, 99.5)),
    xlabel="theoretical F quantile",
    ylabel="observed moderated F",
    title="Q-Q: temporal regulation"
)

plt.show()

In [ ]:
top_temporal = (
    res.loc[temporal]
    .sort_values("F", ascending=False)
    .head(20)
)

display(
    top_temporal[
        ["F", "pvalue", "qvalue"]
    ]
)

In [ ]:
temporal = list(
    res.index[
        res["sig"] == "temporal"
    ]
)

print(
    len(temporal),
    "temporal genes"
)

In [ ]:
clusters = ov.bulk.temporal_clusters(
    lognorm,
    time,
    genes=temporal,
    n_clusters=6,
    m="auto",
    seed=0
)

In [ ]:
clusters.head()

In [ ]:
print(clusters.columns)

In [ ]:
clusters = ov.bulk.temporal_clusters(
    lognorm,
    time,
    genes=temporal,
    n_clusters=6,
    m="auto",
    seed=0
)

In [ ]:
sizes = (
    clusters["cluster"]
    .value_counts()
    .sort_index()
)

print(sizes)

In [ ]:
import pymfuzz
import matplotlib.pyplot as plt

times = sorted(time.unique())

traj = ov.pd.DataFrame({
    t: lognorm.loc[
        temporal,
        time.index[time == t]
    ].mean(axis=1)
    for t in times
})

print(traj.shape)

In [ ]:
z = pymfuzz.standardise(
    pymfuzz.as_expression_matrix(traj)
)

In [ ]:
fc = pymfuzz.mfuzz(
    z,
    c=6,
    m=clusters.attrs["m"],
    random_state=0
)

print(
    f"soft clustering: "
    f"{len(temporal)} temporal genes x "
    f"{len(times)} time points -> "
    f"{fc.n_clusters} fuzzy clusters"
)

In [ ]:
time_labels = [
    f"D{t:g}"
    for t in times
]

pymfuzz.mfuzz_plot(
    z,
    fc,
    mfrow=(2, 3),
    time_labels=time_labels,
    figsize=(11, 6)
)

plt.show()

In [ ]:
centers = clusters.attrs["centers"]

fig, ax = plt.subplots(figsize=(7, 4))

for name, row in centers.iterrows():
    ax.plot(
        times,
        row.values,
        marker="o",
        ms=4,
        label=f"C{name}"
    )

ax.axhline(
    0,
    color="grey",
    lw=0.7,
    ls="--"
)

ax.set_xlabel("Day")
ax.set_ylabel("Standardised expression")
ax.set_title("PF4+PPBP+ Mono temporal expression waves")

ax.legend(
    fontsize=8,
    ncol=3
)

plt.tight_layout()
plt.show()

In [ ]:
order = clusters.sort_values(
    ["cluster", "membership"],
    ascending=[True, False]
).index

zmat = z.to_dataframe().loc[order]

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(
    zmat.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    interpolation="nearest"
)

ax.set_xticks(range(len(times)))
ax.set_xticklabels(
    ["D-1", "D0", "D0.5", "D1", "D2", "D3", "D7", "D10", "D12", "D14", "D17", "D21", "D25", "D28", "D33"],
    rotation=45,
    ha="right"
)

ax.set_xlabel("Time")
ax.set_ylabel(f"{len(order)} temporal genes")
ax.set_title("PF4+PPBP+ Mono temporal genes ordered by Mfuzz cluster")

plt.colorbar(im, ax=ax, label="z-score", shrink=0.6)
plt.tight_layout()
plt.show()

In [ ]:
for c in sorted(clusters["cluster"].unique()):

    top20_genes = (
        clusters.loc[clusters["cluster"] == c]
        .sort_values("membership", ascending=False)
        .head(20)
        .index
        .tolist()
    )

    print(f"\n===== Cluster {c} =====")
    print(top20_genes)

In [ ]:
top20_by_cluster = {}

for c in sorted(clusters["cluster"].unique()):

    top20 = (
        clusters.loc[clusters["cluster"] == c]
        .sort_values("membership", ascending=False)
        .head(20)
        [["cluster", "membership"]]
    )

    top20_by_cluster[c] = top20

    print(f"\n===== Cluster {c} =====")
    display(top20)

In [ ]:
import inspect
import pymfuzz

print(inspect.signature(pymfuzz.Dmin))
print(inspect.signature(pymfuzz.cselection))
print(inspect.signature(pymfuzz.partcoef))

In [ ]:
crange = range(2, 11)
m = clusters.attrs["m"]

In [ ]:
dmin_res = pymfuzz.Dmin(
    z,
    m=m,
    crange=crange,
    repeats=5,
    visu=True,
    random_state=0
)
save_fig("")  
send_plot()

In [ ]:
csel_res = pymfuzz.cselection(
    z,
    m=m,
    crange=crange,
    repeats=5,
    visu=True,
    random_state=0
)

In [ ]:
print(type(dmin_res))
print(dmin_res)

print(type(csel_res))
print(csel_res)

In [ ]:
clusters3 = ov.bulk.temporal_clusters(
    lognorm,
    time,
    genes=temporal,
    n_clusters=3,
    m="auto",
    seed=0
)

print(
    clusters3["cluster"]
    .value_counts()
    .sort_index()
)

print("m =", clusters3.attrs["m"])

Fgiure 3B

In [ ]:
fc3 = pymfuzz.mfuzz(
    z,
    c=3,
    m=clusters3.attrs["m"],
    random_state=0
)

time_labels = [
    f"D{t:g}" for t in times
]

pymfuzz.mfuzz_plot(
    z,
    fc3,
    mfrow=(1, 3),
    time_labels=time_labels,
    figsize=(12, 4)
)
save_fig("PF4+PPBP+ Mono temporal expression waves")  
plt.show()

In [ ]:
centers3 = clusters3.attrs["centers"]

fig, ax = plt.subplots(figsize=(7, 4))

for name, row in centers3.iterrows():
    ax.plot(
        times,
        row.values,
        marker="o",
        ms=4,
        label=f"Cluster {name}"
    )

ax.axhline(
    0,
    color="grey",
    lw=0.7,
    ls="--"
)

ax.set_xlabel("Day")
ax.set_ylabel("Standardised expression")
ax.set_title("PF4+PPBP+ Mono temporal expression waves")
ax.legend()

plt.tight_layout()
save_fig("PF4+PPBP+ Mono temporal expression waves")  
send_plot()
plt.show()

In [ ]:
for c in sorted(clusters3["cluster"].unique()):

    top30 = (
        clusters3.loc[clusters3["cluster"] == c]
        .sort_values("membership", ascending=False)
        .head(30)
    )

    print(f"\n===== Cluster {c} | n={(clusters3['cluster'] == c).sum()} =====")
    print(top30.index.tolist())

In [ ]:
clusters3["cluster"].value_counts().sort_index()

Fgiure 3A

In [ ]:
order = clusters3.sort_values(
    ["cluster", "membership"],
    ascending=[True, False]
).index

zmat = z.to_dataframe().loc[order]

fig, ax = plt.subplots(figsize=(5, 6))

im = ax.imshow(
    zmat.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    interpolation="nearest"
)

ax.set_xticks(range(len(times)))
ax.set_xticklabels(
    [f"D{t:g}" for t in times],
    rotation=45,
    ha="right"
)

ax.set(
    xlabel="Time",
    title="Trajectories ordered by cluster",
    ylabel=f"{len(order)} temporally regulated genes"
)

plt.colorbar(
    im,
    ax=ax,
    label="z-score",
    shrink=0.6
)

plt.tight_layout()

save_fig("PF4+PPBP+ Mono temporal genes heatmap")
plt.show()

Figure 3C_D_E

In [ ]:
import pySTIM as pst
import pySTIM._enrichment as _en
import seaborn as sns

for name, module in [
    ("plt", plt),
    ("sns", sns),
    ("np", np),
    ("pd", pd),
    ("gp", gp),
]:
    if not hasattr(_en, name):
        setattr(_en, name, module)

SAVE_DIR = (
    "/20260922"
)

os.makedirs(SAVE_DIR, exist_ok=True)

SHOW_NUM = 8
show_plot = True

LIBRARIES = {
    "MSigDB_Hallmark_2020": "#E0B1CB",
    "GO_Biological_Process_2023": "#80b1d3",
    "KEGG_2021_Human": "#eeb248",
    "Reactome_2022": "#cb5b3a",
}

temporal_background = list(temporal)

_orig_enrichr = gp.enrichr

def _fixed_enrichr(*args, **kwargs):
    kwargs["organism"] = "human"
    kwargs["background"] = temporal_background
    return _orig_enrichr(*args, **kwargs)

gp.enrichr = _fixed_enrichr
_en.gp.enrichr = _fixed_enrichr
current_pdf_path = None
_real_plt_show = plt.show

def _custom_plt_show(*args, **kwargs):
    global current_pdf_path

    fig = plt.gcf()

    if current_pdf_path:
        fig.savefig(
            current_pdf_path,
            dpi=300,
            format="pdf",
            bbox_inches="tight"
        )
        print(f"已保存: {current_pdf_path}")

    if show_plot:
        _real_plt_show(*args, **kwargs)

    plt.close(fig)

_en.plt.show = _custom_plt_show

for c in sorted(clusters3["cluster"].unique()):

    cluster_df = (
        clusters3[
            clusters3["cluster"] == c
        ]
        .sort_values(
            "membership",
            ascending=False
        )
    )

    genes = cluster_df.index.astype(str).tolist()

    print("\n" + "=" * 70)
    print(
        f"Temporal Cluster {c} | "
        f"{len(genes)} genes"
    )
    print("=" * 70)

    for lib_name, color in LIBRARIES.items():

        clean_lib = lib_name.replace(" ", "_")

        current_pdf_path = os.path.join(
            SAVE_DIR,
            f"PF4_Temporal_Cluster{c}_{clean_lib}.pdf"
        )

        try:
            pst.gesa(
                genes,
                enrichr_library=lib_name,
                show_number=SHOW_NUM,
                bar_color=color,
            )

        except Exception as e:
            print(
                f"[Cluster {c} - {lib_name}] "
                f"富集失败或无显著结果: {e}"
            )



_en.plt.show = _real_plt_show

print("\n完成")
print(SAVE_DIR)